# 01 — CIC-IDS2017 flows: exploration and timestamp check

CIC-IDS2017 records five working days (3–7 July 2017) of a small enterprise network. Each row is a network flow with its endpoints, a timestamp, about 80 statistics and a label. This notebook checks the data before modelling, with a focus on the timestamps, because the graph windows and both evaluation protocols depend on them.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
cfg = yaml.safe_load(Path('configs/experiment.yaml').read_text())
metrics = json.loads(Path('results/metrics.json').read_text())
from src.data import load_cic, find_column
flows = load_cic(cfg['data']['path'])
print(f"{len(flows):,} labelled flows, {int(flows['_target'].sum()):,} attacks ({flows['_target'].mean():.1%})")

## Attack types per day

Each attack type happens on a single day. This is why a purely chronological split (protocol A) tests attacks that were never seen in training.

In [ ]:
table = pd.crosstab(flows['_attack_type'], flows['_day']).reindex(columns=['Monday','Tuesday','Wednesday','Thursday','Friday'], fill_value=0)
display(table.sort_values(list(table.columns), ascending=False))

## Timestamp repair

Several CSV files write afternoon times on a 12-hour clock without AM/PM (for example `7/7/2017 3:56` for 15:56). The loader shifts every hour before 08:00 by 12 hours, since captures only cover working hours. The table below compares the repaired attack times with the schedule published by the dataset authors; matching windows confirm the repair.

In [ ]:
documented = {
 ('Tuesday','FTP-Patator'):'09:20–10:20', ('Tuesday','SSH-Patator'):'14:00–15:00',
 ('Wednesday','DoS slowloris'):'09:47–10:10', ('Wednesday','DoS Slowhttptest'):'10:14–10:35', ('Wednesday','DoS Hulk'):'10:43–11:00',
 ('Wednesday','DoS GoldenEye'):'11:10–11:23', ('Wednesday','Heartbleed'):'15:12–15:32',
 ('Thursday','Web Attack - Brute Force'):'09:20–10:00', ('Thursday','Web Attack - XSS'):'10:15–10:35', ('Thursday','Web Attack - Sql Injection'):'10:40–10:42',
 ('Thursday','Infiltration'):'14:19–15:45', ('Friday','Bot'):'10:02–11:02', ('Friday','PortScan'):'13:55–14:35', ('Friday','DDoS'):'15:56–16:16'}
sched = pd.DataFrame(metrics['dataset']['attack_schedule_after_repair'])
sched['documented (CIC)'] = [documented.get((d, a), '–') for d, a in zip(sched['_day'], sched['_attack_type'])]
sched = sched.rename(columns={'_day':'day','_attack_type':'attack','min':'first flow','max':'last flow','size':'flows'})
display(sched)
display(Image(filename='results/timeline.png'))

## Top talkers

IP addresses are used only to build the graph (hosts become nodes); they never enter the model as numeric features.

In [ ]:
display(flows['Source IP'].value_counts().head(8).rename('outgoing flows').to_frame())
display(flows.loc[flows['_target']==1, 'Source IP'].value_counts().head(5).rename('outgoing attack flows').to_frame())

## Heavy-tailed features

Counters such as bytes, packets and durations span many orders of magnitude. The preprocessing applies `log1p` to them, then standard scaling, fitted on training flows only.

In [ ]:
cols = ['Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Flow Bytes/s']
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, c in zip(axes, cols):
    v = flows[c].replace([np.inf, -np.inf], np.nan).dropna().clip(lower=0)
    ax.hist(np.log1p(v), bins=60, color='#2a78d6'); ax.set_title(f'log1p({c})', fontsize=9); ax.set_yscale('log')
plt.tight_layout()

## Splits used by the experiments

In [ ]:
rows = []
for proto in ('protocol_a', 'protocol_b'):
    for part, s in metrics[proto]['splits'].items():
        rows.append({'protocol': proto, 'split': part, 'flows': s['flows'], 'attacks': s['attacks'], 'windows': s['windows'],
                     'attack types': ', '.join(k for k in s['attack_types'] if k != 'Benign')})
display(pd.DataFrame(rows))